# Analyze and plot top PLP covariates

This notebook shows how to access, analyze and plot PLP covariates.

## Setup

Install libraries:

In [ ]:
!pip install numpy
!pip install seaborn

Import libraries:

In [ ]:

import pandas as pd
import seaborn as sns
import os
import matplotlib.pyplot as plt
import re

Set pandas option to display all rows and columns

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

## Define input and output directories, number of covariates to analyze, filters

In [ ]:
# Path to your folder with PLP results, adjust as needed
download_path = '~/Downloads'

# PLPs to analyze, adjust as needed
plp_dict = {
    '3y_llr': 'plp-5249860966-f9drp',
    '5y_llr': 'plp-5054926538-4v4jc',
    '10y_llr': 'plp-9987482270-qckr7'
}

# PNG and PDF plots destination folder, adjust as needed
outdir = '../figures/'
# SVG plots destination folder, adjust as needed
svg_outdir = '../figures/SVG backups/'
os.makedirs(outdir, exist_ok=True)
os.makedirs(svg_outdir, exist_ok=True)

# Data destination folder, adjust as needed
data_outdir = '../data/'
os.makedirs(data_outdir, exist_ok=True)

# How many top covariates to analyze and plot, adjust as needed
n_cov = 15

# Covariate filters, adjust as needed
filter_a = 'Deprecated'
filter_b = 'Unknown concept'
filter_c = 'index year'

# Variable to strore top covariates for each model
top_covariates_dict = {}

## Helper functions to extract categories and subcategories from the covariates names

In [ ]:
def extract_category(name):
    if '=' in name:
        return name.split('=')[0].strip()
    m = re.search(r'(.+?) during', name)
    if m:
        return m.group(1).strip()
    return None

def extract_subcategory(name):
    if '=' in name:
        return name.split('=')[1].strip()
    m = re.search(r':\s*(.+)', name)
    if m:
        return m.group(1).strip()
    return None

## Analyze PLPs, plot results and save common covariates

In [ ]:
# Process and plot top covariates
for model, plp in plp_dict.items():
    plp_path = os.path.join(os.path.expanduser(download_path), plp)
    plp_outfile = os.path.join(f'covariateImportance_{model}_{plp}')
    data_path = os.path.join(data_outdir, f'{plp_outfile}.csv')
    png_path = os.path.join(outdir, f'{plp_outfile}.png')
    pdf_path = os.path.join(outdir, f'{plp_outfile}.pdf')
    svg_path = os.path.join(svg_outdir, f'{plp_outfile}.svg')
    covariates_path = f'{plp_path}/plp_outputs/lasso_logistic_regression/plpResult/model/covariateImportance.csv'
    covar_importances = pd.read_csv(covariates_path)
    
    # Process covariates
    covar_importances = covar_importances[~covar_importances.covariateName.str.contains(filter_b)]
    covar_importances = covar_importances[~covar_importances.covariateName.str.contains(filter_a)]
    covar_importances = covar_importances[~covar_importances.covariateName.str.contains(filter_c)]
    covar_importances.sort_values(by='covariateValue', ascending=False, inplace=True)
    
    # Save covariates
    covar_importances.to_csv(data_path)
    print(f'Top {n_cov} covariates for {model} saved in: {data_path}')
    
    # Plot top covariates
    plt.figure(figsize=(10,n_cov/2))
    sns.set(font_scale=2)
    plt.xlabel('covariateValue', fontweight='bold')
    plt.ylabel('covariateName', fontweight='bold')
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['axes.labelweight'] = 'bold'
    #plt.yticks(fontsize=24)    
    sns.barplot(x='covariateValue',
                y='covariateName',
                data=covar_importances[:n_cov]
            )
    
    for plot_path in (png_path, pdf_path, svg_path):
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        print(f'Plot for {model} saved in: {plot_path}')

    # Store top covariates for each workflow (name and value)
    top_covs = covar_importances[['covariateName', 'covariateValue']].head(n_cov)
    top_covariates_dict[model] = dict(zip(top_covs['covariateName'], top_covs['covariateValue']))

# Find common covariates across all workflows
common_covariates = set.intersection(*(set(d.keys()) for d in top_covariates_dict.values()))

# Save common covariates with average and each workflow value
common_cov_list = []
for covariate in common_covariates:
    values = [top_covariates_dict[model][covariate] for model in plp_dict]
    avg_value = sum(values) / len(values)
    row = [covariate] + values + [avg_value]
    common_cov_list.append(row)

columns = ['Covariate'] + [f'{model}' for model in plp_dict] + ['Average']
common_cov_df = pd.DataFrame(common_cov_list, columns=columns)
common_cov_df.sort_values(by='Average', ascending=False, inplace=True)

common_cov_df['Category'] = common_cov_df['Covariate'].apply(extract_category)
common_cov_df['Subcategory'] = common_cov_df['Covariate'].apply(extract_subcategory)

common_cov_path = os.path.join(data_outdir, f'common_covariates_from_top{n_cov}.csv')
common_cov_df.to_csv(common_cov_path, index=False)
print(f'Common covariates saved in: {data_outdir}')

print(f'\nNumber of common covariates: {len(common_covariates)}')
common_cov_df